In [17]:
from datetime import datetime
import openmeteo_requests

In [18]:
class IncreaseSpeed:

    def __init__(self, current_speed: int, max_speed: int, step=10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed < self.max_speed:
            diff = min(self.step, self.max_speed - self.current_speed)
            self.current_speed += diff
            return self.current_speed, diff
        else:
            raise StopIteration

In [19]:
class DecreaseSpeed:

    def __init__(self, current_speed: int, min_speed: int, step=10):
        self.current_speed = current_speed
        self.min_speed = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed > self.min_speed:
            diff = min(self.step, self.current_speed - self.min_speed)
            self.current_speed -= diff
            return self.current_speed, diff
        else:
            raise StopIteration

In [20]:
class Car:

    _total_cars_on_road = 0

    def __init__(self, max_speed: int, current_speed=0):
        self.max_speed = max_speed
        self.current_speed = current_speed

        if self.current_speed > 0:
            self.state = 'road'
            Car._total_cars_on_road += 1
        else:
            self.state = 'parking'

    def accelerate(self, upper_border=None, step=10):
        
        if self.state == 'parking':
            self.state = 'road'
            Car._total_cars_on_road += 1

        start_speed = self.current_speed

        if upper_border is not None and isinstance(upper_border, (int, float)) and upper_border > self.current_speed:
            limit = min(upper_border, self.max_speed)

            accelerator = IncreaseSpeed(self.current_speed, limit, step)
            
            for speed, diff in accelerator:
                print(f'INFO: Speed increases by {diff}')
                self.current_speed = speed
        else:

            accelerator = IncreaseSpeed(self.current_speed, self.max_speed, step)
            try:
                speed, diff = next(accelerator)
                print(f'INFO: Speed increases by {diff}')
                self.current_speed = speed
            except StopIteration:
                pass

        print(f'INFO: The speed of this car has been increased from {start_speed} to {self.current_speed}')

    def brake(self, lower_border=None, step=10):
        start_speed = self.current_speed

        if lower_border is not None and isinstance(lower_border, (int, float)) and lower_border < self.current_speed:

            limit = max(lower_border, 0)

            #    :)
            retarder = DecreaseSpeed(self.current_speed, limit, step)

            for speed, diff in retarder:
                print(f'INFO: Speed decreases by {diff}')
                self.current_speed = speed

        else:

            retarder = DecreaseSpeed(self.current_speed, 0, step)
            try:
                speed, diff = next(retarder)
                print(f'INFO: Speed decreases by {diff}')
                self.current_speed = speed
            except StopIteration:
                pass

        print(f'INFO: The speed of this car has been decreased from {start_speed} to {self.current_speed}')

    def parking(self):
        if self.state == 'road':
            self.state = 'parking'
            self.brake(0)
            Car._total_cars_on_road -= 1
            print('Parking the car...')
        else:
            print('Car is already parked.')


    @classmethod
    def total_cars(cls):
        return Car._total_cars_on_road

    @staticmethod
    def show_weather():
        openmeteo = openmeteo_requests.Client()
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": 59.9386,
            "longitude": 30.3141,
            "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
            "wind_speed_unit": "ms",
            "timezone": "Europe/Moscow"
        }
        response = openmeteo.weather_api(url, params=params)[0]

        # The order of variables needs to be the same as requested in params->current!
        current = response.Current()
        current_temperature_2m = current.Variables(0).Value()
        current_apparent_temperature = current.Variables(1).Value()
        current_rain = current.Variables(2).Value()
        current_wind_speed_10m = current.Variables(3).Value()

        print(f"Current time: {datetime.fromtimestamp(current.Time()+response.UtcOffsetSeconds())} {response.TimezoneAbbreviation().decode()}")
        print(f"Current temperature: {round(current_temperature_2m, 0)} C")
        print(f"Current apparent_temperature: {round(current_apparent_temperature, 0)} C")
        print(f"Current rain: {current_rain} mm")
        print(f"Current wind_speed: {round(current_wind_speed_10m, 1)} m/s")


In [21]:
car1 = Car(100, 20) # max_speed = 100, initial speed = 5
car2 = Car(60, 30) # max_speed = 60, initial speed = 30
car3 = Car(100, 0) # a car that is off road upon creation
print(f"Total cars on road: {Car.total_cars()}")

Total cars on road: 2


In [22]:
car1.accelerate(100)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


In [23]:
car2.accelerate(50)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50


In [24]:
print("Speed of car 1:", car1.current_speed)

Speed of car 1: 100


In [25]:
print("Speed of car 2:", car2.current_speed)

Speed of car 2: 50


In [26]:
car1.brake(10)

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


In [27]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Parking the car...
Total cars on road: 1


In [28]:
car3.accelerate(80)# car3 is now on the road
car3.show_weather()
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 80
Current time: 2026-03-20 00:30:00 GMT+3
Current temperature: 2.0 C
Current apparent_temperature: -2.0 C
Current rain: 0.0 mm
Current wind_speed: 5.0 m/s
Total cars on road: 2


In [29]:
car2.accelerate(10) # # car2 goes from parking on the road
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3


In [30]:
Car.show_weather()

Current time: 2026-03-20 00:30:00 GMT+3
Current temperature: 2.0 C
Current apparent_temperature: -2.0 C
Current rain: 0.0 mm
Current wind_speed: 5.0 m/s
